# CCLE 2019
The Cancer Cell Line Encyclopedia (CCLE) is a resource of human cancer cell lines by different tissues. Cell lines are collected and made proliferate indefinitely in vitro.

CCLE was developed primarily by the Broad Institute in collaboration with Novartis and others to support cancer biology research and drug discovery.

The cell line study supports large-scale drug screening and genomic studies. The latter is because cells proliferate over time and we can observe a significant genetic drift. However, a key limitation is the absence of the tumor microenvironment. Despite this, CCLE is widely used in cancer research and precision medicine.



## Imports

In [ ]:
!pip install combat

In [3]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.manifold import TSNE
from scipy.stats import skew, kurtosis
from sklearn.decomposition import PCA
from pandas.plotting import scatter_matrix
from sklearn.preprocessing import StandardScaler
import umap
from combat.pycombat import pycombat
from sklearn.model_selection import train_test_split
import warnings


warnings.filterwarnings('ignore')

## Roadmap

In this notebook we will analyse two different cell-lines:

Cancer Cell Line (Broad, 2019) [2]
We obtained expression and mutation data in the cBio Portal by querying CCLE data and then download all the files of the study listed above, more specifically:

data_mutations.txt : Dataframe containing information of all the mutated genes in the samples;
mut_query.txt: information of all the samples, their study, and the actual mutations;
mRNA_data.txt: expression data with samples as columns and genes as rows;
NOTE: for each study the name of the file above is slighty different, we will provide an example of the data

The roadmap we will take for each dataset is the following:

Description of the methodology
Sample Analysis: analyse QC metrics and filter sample and try to get a grasp of how the data look like
Gene Analysis: analyse QC metrics and filter genes and try to get a grasp of how the data look like
Dimensionality reduction: plot data in 2 dimensions with various dimensionality reduction techniques and see if data linearly separable or see possible batch effects
Finally, we will prepare data for classification by merging the two datasets by understanding whether some batch effects are present and how to correct them.



## Example Data

In [ ]:
example_data_mutations = pd.read_csv('/Users/andreafabbricatore/Desktop/mac_bocconi/MAI/year 1/semester 2/ML Lab/AILAB/DATA/ccle_2019/data_mutations.txt',
                                      sep = '\t',
                                      comment = '#')
example_data_mutations.head()

In [ ]:
example_mut_query = pd.read_csv('/home/3173616/ML_lab/cellline_ccle_broad/mutations_ccle_2012.txt',
                    sep = '\t',
                    comment = '#')
example_mut_query.head()

In [ ]:
example_mRNA = pd.read_csv('/Users/andreafabbricatore/Desktop/mac_bocconi/MAI/year 1/semester 2/ML Lab/AILAB/DATA/tcga/data_mrna_seq_fpkm.txt',
                    sep = '\t',
                    comment = '#')
example_mRNA.head()

In [ ]:
# Free memory
del example_data_mutations, example_mRNA

## Utils

In [4]:
def plot_violin_metrics(data, axis):
    """  
    Input:
        data (pd.Dataframe): gene expression matrix, rows are genes and samples are columns
        axis (int): 0 samples, 1 genes
    Output:
        Plot violin plot samples/genes wide 
    """

    if axis == 1:
        
        # Total expression 
        total_counts_genes = data.sum(axis=1).reset_index(drop=True)
        # Mean expression
        mean_values_genes = data.mean(axis=1).reset_index(drop=True)
        # Median expression
        median_values_genes = data.median(axis=1).reset_index(drop=True)
        # Standard deviation of expression
        std_values_genes = data.std(axis = 1).reset_index(drop = True)
        # Coefficient of variation of expression
        cv_values_genes = std_values_genes /mean_values_genes
        # Number of non-zero element 
        total_features_genes = data.astype(bool).sum(axis = 1).reset_index(drop = True)

        # Set up subplots
        fig, ax = plt.subplots(1, 6, figsize=(25, 12))

        # Total Counts per Gene
        sns.violinplot(y=total_counts_genes, ax=ax[0], color='firebrick')
        ax[0].set_title('Total Counts Across Genes')
        ax[0].set_xlabel('')
        ax[0].grid(True)

        # Mean Counts per Gene
        sns.violinplot(y=mean_values_genes, ax=ax[1], color='steelblue')
        ax[1].set_title('Mean Counts Across Genes')
        ax[1].set_xlabel('')
        ax[1].grid(True)

        # Median Counts per Gene
        sns.violinplot(y=median_values_genes, ax=ax[2], color='orange')
        ax[2].set_title('Median Counts Across Genes')
        ax[2].set_xlabel('')
        ax[2].grid(True)

        # Std values per Gene
        sns.violinplot(y=std_values_genes, ax=ax[3], color='darkgreen')
        ax[3].set_title('Standard deviation across Genes')
        ax[3].set_xlabel('')
        ax[3].grid(True)

        # Coefficeint of variation std/mean per Gene
        sns.violinplot(y=cv_values_genes, ax=ax[4], color='indigo')
        ax[4].set_title('Coefficient of variation across Genes')
        ax[4].set_xlabel('')
        ax[4].grid(True)

        # Total Features per Gene
        sns.violinplot(y=total_features_genes, ax=ax[5], color='silver')
        ax[5].set_title('Total Features Across Genes')
        ax[5].set_xlabel('')
        ax[5].grid(True)

        plt.tight_layout()
        plt.show()

        # Print differences between extremes
        print(f'Max_total_count ({total_counts_genes.max()}) - Min_total_count ({total_counts_genes.min()}) = {total_counts_genes.max() - total_counts_genes.min()}')
        print(f'Max_mean_count ({mean_values_genes.max()}) - Min_mean_count ({mean_values_genes.min()}) = {mean_values_genes.max() - mean_values_genes.min()}')
        print(f'Max_median_count ({median_values_genes.max()}) - Min_median_count ({median_values_genes.min()}) = {median_values_genes.max() - median_values_genes.min()}')
        print(f'Max std  ({std_values_genes.max()}) - Min std ({std_values_genes.min()}) = {std_values_genes.max() - std_values_genes.min()}')
        print(f'Max cv  ({cv_values_genes.max()}) - Min cv ({cv_values_genes.min()}) = {cv_values_genes.max() - cv_values_genes.min()}')
        print(f'Max feature count  ({total_features_genes.max()}) - Min feature count ({total_features_genes.min()}) = {total_features_genes.max() - total_features_genes.min()}')

    
    if axis == 0:

        total_counts = data.sum(axis=0).reset_index(drop=True)
        mean_values = data.mean(axis=0).reset_index(drop=True)
        std_values = data.std(axis = 0).reset_index(drop=True)
        median_values = data.median(axis=0).reset_index(drop=True)
        cv_values = std_values/mean_values
        total_features = data.astype(bool).sum(axis = 0).reset_index(drop = True)


        # Set up the figure with 3 subplots
        fig, ax = plt.subplots(1, 6, figsize=(25, 12))

        # Total Counts
        sns.violinplot(y=total_counts, ax=ax[0], color='firebrick')
        ax[0].set_title('Total Counts Across Samples')
        ax[0].set_xlabel('')
        ax[0].grid(True)

        # Mean Values
        sns.violinplot(y=mean_values, ax=ax[1], color='steelblue')
        ax[1].set_title('Mean Across Samples')
        ax[1].set_xlabel('')
        ax[1].grid(True)

        # Median Values
        sns.violinplot(y=median_values, ax=ax[2], color='orange')
        ax[2].set_title('Median Samples')
        ax[2].set_xlabel('')
        ax[2].grid(True)

        # Std values
        sns.violinplot(y=std_values, ax=ax[3], color='darkgreen')
        ax[3].set_title('Standard deviation Samples')
        ax[3].set_xlabel('')
        ax[3].grid(True)

        # Coefficeint of variation std/mean per sample
        sns.violinplot(y=cv_values, ax=ax[4], color='indigo')
        ax[4].set_title('Coefficient of variation across Samples')
        ax[4].set_xlabel('')
        ax[4].grid(True)

        # Total features 
        sns.violinplot(y=total_features, ax=ax[5], color='silver')
        ax[5].set_title('Total features across Samples')
        ax[5].set_xlabel('')
        ax[5].grid(True)


        plt.tight_layout()
        plt.show()

        # Print differences between extremes
        print(f'Max_total_count ({total_counts.max()}) - Min_total_count ({total_counts.min()}) = {total_counts.max() - total_counts.min()}')
        print(f'Max_mean_count ({mean_values.max()}) - Min_mean_count ({mean_values.min()}) = {mean_values.max() - mean_values.min()}')
        print(f'Max_median_count ({median_values.max()}) - Min_median_count ({median_values.min()}) = {median_values.max() - median_values.min()}')
        print(f'Max_std ({std_values.max()}) - Min_std ({std_values.min()}) = {std_values.max() - std_values.min()}')
        print(f'Max CV ({cv_values.max()}) - Min CV ({cv_values.min()}) = {cv_values.max() - cv_values.min()}')
        print(f'Max feature count  ({total_features.max()}) - Min feature count ({total_features.min()}) = {total_features.max() - total_features.min()}')


def plot_skewness_kurtosis(data,axis):

    """  
    Input:
        data (pd.Dataframe): gene expression matrix, rows are genes and samples are columns
        axis (int): 0 samples, 1 genes
    Output:
        Plot skewness, kurtosis plot samples/genes wide 
    """


    if axis == 0:
        # Skewness
        skewness_per_sample = data.apply(skew, axis=0)
        # Kurtosis
        kurtosis_per_sample = data.apply(kurtosis, axis=0)

        # Plot skewness
        plt.figure(figsize=(14, 5))
        plt.subplot(1, 2, 1)
        plt.bar(range(len(skewness_per_sample)), skewness_per_sample.values, color='mediumpurple')
        plt.title("Skewness of Expression per Sample")
        plt.xlabel("Sample Index")
        plt.ylabel("Skewness")
        plt.grid(True)

        # Plot kurtosis
        plt.subplot(1, 2, 2)
        plt.bar(range(len(kurtosis_per_sample)), kurtosis_per_sample.values, color='darkorange')
        plt.title("Excess Kurtosis per Sample")
        plt.xlabel("Sample Index")
        plt.ylabel("Kurtosis")
        plt.grid(True)

        plt.tight_layout()
        plt.show()
        
    if axis == 1:
        # Compute skewness and kurtosis per sample (column-wise)
        skewness_per_gene = data.apply(skew, axis=1)
        kurtosis_per_gene = data.apply(kurtosis, axis=1)
        
        # Plot skewness
        plt.figure(figsize=(14, 5))
        plt.subplot(1, 2, 1)
        plt.bar(range(len(skewness_per_gene)), skewness_per_gene.values, color='mediumpurple')
        plt.title("Skewness of Expression per Gene")
        plt.xlabel("Sample Index")
        plt.ylabel("Skewness")
        plt.grid(True)
        
        # Plot kurtosis
        plt.subplot(1, 2, 2)
        plt.bar(range(len(kurtosis_per_gene)), kurtosis_per_gene.values, color='darkorange')
        plt.title("Excess Kurtosis per Gene")
        plt.xlabel("Sample Index")
        plt.ylabel("Kurtosis")
        plt.grid(True)
        
        plt.tight_layout()
        plt.show()

def heatmap_plot(title,data,samples):
    """
    Input:
        title (str): Title of our plot
        data (pd.Dataframe): gene expression matrix, rows are genes and samples are columns
        samples (list or np.array): subsample to plot the heatmap on
    Output:
        Pearson correlation heatmap between samples 
    """
    plt.title(title)
    sns.heatmap(data.loc[:, samples].corr(), cmap = 'viridis', fmt ='.01f');
    print(f"Number of samples plotted: {(len(samples),len(samples))}")


def plot_PCA_tsne_umap(data_ordered, mutated_samples, non_mutated_samples, title, categories):
    """
    Generates a combined plot of PCA, t-SNE, and UMAP visualizations for given data.

    Parameters:
    - data_ordered: pandas DataFrame or array-like, features x samples matrix
    - mutated_samples: list of sample identifiers for mutated samples
    - non_mutated_samples: list of sample identifiers for non-mutated samples
    - title: str, overall title for the figure
    - categories: tuple of two strings (mutated_label, non_mutated_label)
    """
    # Build labels and colors
    mutation_status = [categories[0]] * len(mutated_samples) + [categories[1]] * len(non_mutated_samples)
    label_map = {categories[0]: 0, categories[1]: 1}
    color_values = [label_map[label] for label in mutation_status]

    # Convert data: transpose to samples x features
    X = data_ordered.T.to_numpy() 

    # PCA
    pca = PCA(n_components=2, random_state=42)
    pca_result = pca.fit_transform(X)

    # t-SNE
    tsne = TSNE(n_components=2, random_state=42, learning_rate='auto', init='random')
    tsne_result = tsne.fit_transform(X)

    # UMAP
    reducer = umap.UMAP(n_components=2, random_state=42)
    umap_result = reducer.fit_transform(X)

    # Plotting
    plt.figure(figsize=(18, 5))

    # PCA subplot
    plt.subplot(1, 3, 1)
    plt.scatter(pca_result[:, 0], pca_result[:, 1], c=color_values, cmap='viridis', alpha=0.8)
    plt.title('PCA: {} vs {}'.format(categories[0], categories[1]))
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    plt.grid(True)

    # t-SNE subplot
    plt.subplot(1, 3, 2)
    plt.scatter(tsne_result[:, 0], tsne_result[:, 1], c=color_values, cmap='viridis', alpha=0.8)
    plt.title('t-SNE: {} vs {}'.format(categories[0], categories[1]))
    plt.xlabel('tSNE1')
    plt.ylabel('tSNE2')
    plt.grid(True)

    # UMAP subplot
    plt.subplot(1, 3, 3)
    plt.scatter(umap_result[:, 0], umap_result[:, 1], c=color_values, cmap='viridis', alpha=0.8)
    plt.title('UMAP: {} vs {}'.format(categories[0], categories[1]))
    plt.xlabel('UMAP1')
    plt.ylabel('UMAP2')
    plt.grid(True)

    # Legend and overall title
    handles = [plt.Line2D([], [], marker='o', linestyle='', color=plt.cm.viridis(label_map[c]/1), label=c)
               for c in categories]
    plt.legend(handles=handles, title='Category', loc='upper right')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def standard(df):
    """
    Input:
        df (pandas.dataframe): dataframe rows genes and columns samples.

    Output:
        dt (pd.dataframe): data after standardization of the samples.

    """
    dt = df.copy()
    #dt = dt.astype("float64")
    # function for data standardization
    cell_names = dt.columns
    gene_names = dt.index

    scaler = StandardScaler()
    scaler.fit(dt)
    dt = scaler.transform(dt)

    dt = pd.DataFrame(dt)
    dt.index = gene_names
    dt.columns = cell_names

    return(dt)


## Description
blah blha


## Data Collection

In [ ]:
# ---------mRNA---------
# Description:
# Expression Data with gene as index and cells/samples as columns
mRNA_2019 = pd.read_csv('/home/3173616/ML_lab/ccle_broad_2019/data_mrna_seq_rpkm.txt',
                    sep = '\t',
                    comment = '#')
mRNA_2019.set_index('Hugo_Symbol',inplace=True)
# Merge with mean duplicated rows
mRNA_2019 = mRNA_2019.groupby(mRNA_2019.index).mean()


#--------Mut query----------
# Description:
# Data telling  which sample is mutated and not-mutated
mut_query_2019= pd.read_csv('/home/3173616/ML_lab/ccle_broad_2019/mutations_ccle_2019.txt',
                    sep = '\t',
                    comment = '#')
mut_query_2019.set_index('SAMPLE_ID',inplace=True)

# ------Mut ALL-----------
# Dataset containing mutation classes for all genes, it only contains mutated samples
mut_all_2019 = pd.read_csv('/home/3173616/ML_lab/ccle_broad_2019/data_mutations.txt',
                    sep = '\t',
                    comment = '#')
# Extract TP53 from all genes
mut_all_2019 = mut_all_2019[mut_all_2019['Hugo_Symbol'] == 'TP53']
# Remove unwanted information
mut_all_2019 = mut_all_2019[['Tumor_Sample_Barcode', 'Variant_Type']]
mut_all_2019.set_index('Tumor_Sample_Barcode', inplace=True)
# There are repetitions of my mutation type (Variant_Type)
# if there is the same sample with different Variant_Type it should be removed
variant_check = mut_all_2019.groupby(mut_all_2019.index)["Variant_Type"].nunique()

# Consistent mutations
non_consistent_indices = variant_check[variant_check != 1].index
consistent_indices = variant_check[variant_check == 1].index

# Filter 
mut_all_2019 = mut_all_2019.loc[consistent_indices]
mRNA_2019 = mRNA_2019.drop(columns=[col for col in non_consistent_indices if col in mRNA_2019.columns])
mut_query_2019 = mut_query_2019.drop(index = [idx for idx in non_consistent_indices if idx in mut_query_2019.index])

# There are duplicated samples with same Variant Classification
# since it is the same sample just remove the rest and keep one
mut_all_2019 = mut_all_2019[~mut_all_2019.index.duplicated(keep='first')]
# Merged mutated samples
merged_2019 = mut_query_2019.join(mut_all_2019, how = 'left').fillna('WT')
# Free some memory
del mut_all_2019, mut_query_2019, variant_check, consistent_indices, non_consistent_indices


# Take common samples
common = list(set(mRNA_2019.columns) & set(merged_2019.index))
mRNA_2019 = mRNA_2019[common]
merged_2019 = merged_2019.loc[common]

In [ ]:
# Rename columns putting study id
mRNA_2019.rename(columns={i:i + '_ccle_broad_2019' for i in mRNA_2019.columns},inplace=True)
mut_all_2019.rename(index={i:i + '_ccle_broad_2019' for i in mut_all_2019.index},inplace=True)

## Sample filtering

In [ ]:
# Shape of mRNA
print(f'Number of genes: {len(mRNA_2019.index)}\nNumber of samples: {len(mRNA_2019.columns)}')
print(f'Zero entries: {(mRNA_2019 <= 0).sum().sum()}') # entries all positive no zeros

Given the technique bulk mRNA, there is way more sparsity in the data

In [ ]:
# Plot QC metrics
plot_violin_metrics(mRNA_2019,0)

The quality metrics seems pretty well behaved. We can observe RPKM normalisation by the high number of counts. However, the Coefficient of Variation is pretty high for some samples so it's okay to filter for too much variated data.

In [ ]:
# Filter out genes with a coefficient of variation more than 55
cv_genes = mRNA_2019.std(axis = 0)/mRNA_2019.mean(axis = 0) 
samples_to_keep = cv_genes[cv_genes < 55].index

# Filwe
mRNA_2019 = mRNA_2019[samples_to_keep]
mut_all_2019 = mut_all_2019.loc[samples_to_keep]

In [ ]:
corr_matrix = mRNA_2019.corr()
print('Minimum correlation between samples:',corr_matrix.describe().loc['min'].min())
print('Minimum mean correlation between samples:',corr_matrix.describe().loc['mean'].min())

In [ ]:
mutated_samples = mut_all_2019[mut_all_2019['Variant_Type'] != 'WT'].index
non_mutated_samples = mut_all_2019[mut_all_2019['Variant_Type'] == 'WT'].index


random_samples_mut_100 = np.random.choice(mutated_samples,size = 100)
random_samples_non_mut_100 = np.random.choice(non_mutated_samples,size = 100)
random_samples_mut_50_non_mut50 = list(random_samples_mut_100[:50]) + list(random_samples_non_mut_100[:50])

Let us see how the mutations are distributed

In [ ]:
# Sample counts
counts = [len(mutated_samples), len(non_mutated_samples)]

# Create subplots
fig, axs = plt.subplots(2, 1, figsize=(7, 10))

# ---- Plot 1: Sample Counts by Dataset ----
axs[0].bar(x=['Mutated', 'Non Mutated'], height=counts, color=['lightblue', 'orange'])
for i, count in enumerate(counts):
    axs[0].text(i, count + 1, str(count), ha='center', va='bottom', fontsize=12)

axs[0].set_title("Mutated Counts by Datasets")
axs[0].set_ylabel("Number of Samples")
axs[0].grid(True)

# ---- Plot 2: Mutation Counts ----
mut = merged_2019.Variant_Type.value_counts()
axs[1].bar(mut.index, mut.values, color='lightgreen')
for i, val in enumerate(mut.values):
    axs[1].text(i, val + 1, str(val), ha='center', va='bottom', fontsize=10)

axs[1].set_title("Mutation Counts by Type")
axs[1].set_ylabel("Number of Samples")
axs[1].set_xticklabels(mut.index, rotation=45, ha='right')
axs[1].grid(True)

# Adjust layout
plt.tight_layout()
plt.show()

Let us analyse now deeply the samples and try to see if we can find correlation in them.

In [ ]:
heatmap_plot('Correlation for mutated samples',mRNA_2019,random_samples_mut_100)

In [ ]:
heatmap_plot('Correlation for non mutated samples',mRNA_2019,random_samples_non_mut_100)

In [ ]:
heatmap_plot('Correlation between mutated and non mutated samples',mRNA_2019,random_samples_mut_50_non_mut50)

In [ ]:
sample_ids = random_samples_mut_100[:5].tolist() + random_samples_non_mut_100[:5].tolist()
sample_mut_non_mut = mRNA_2019.loc[:, sample_ids]

# Rename columns to clearer labels 
new_labels = ['Mutated'] * 5 + ['Not Mutated'] * 5
sample_mut_non_mut.columns = [f"{label}_{i+1}" for i, label in enumerate(new_labels)]
axes = scatter_matrix(sample_mut_non_mut, diagonal='kde', figsize=(15, 12), color='purple')

# Rotate x and y axis labels for all subplots
for ax in axes.ravel():
    if ax is not None:
        ax.set_xlabel(ax.get_xlabel(), rotation=45, ha='right')
        ax.set_ylabel(ax.get_ylabel(), rotation=0, ha='right')

plt.suptitle("Scatter Matrix: Mutated vs Not Mutated Samples", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
#-----Mutated samples-----------
print('----Mutated Samples-----')
print('1) Minimum correlation between samples:',mRNA_2019.loc[:, mutated_samples].corr().describe().loc['min'].min())
print('2) Minimum mean correlation between samples:',mRNA_2019.loc[:, mutated_samples].corr().describe().loc['mean'].min())
#----- Non Mutated samples-----------
print('----Non Mutated Samples-------')
print('1) Minimum correlation between samples:',mRNA_2019.loc[:, non_mutated_samples].corr().describe().loc['min'].min())
print('2) Minimum mean correlation between samples:',mRNA_2019.loc[:, non_mutated_samples].corr().describe().loc['mean'].min())

In [ ]:
plot_skewness_kurtosis(mRNA_2019,0)

Here both the skewness and the kurtosis are positive. Firstly, positive skewness tells that we have higher values and higher expression for genes, this is expected given the RPKM normalisation. Secondly, positive kurtosis implies more peaked distribution with fatter tails.

## Gene Filtering

In [ ]:
plot_violin_metrics(mRNA_2019,1)

The QC matrix tells us that there are very peaked genes, but the reason is that there are many zero count genes. Our aim is to reduce variation, so we will remove them

In [ ]:
threshold = mRNA_2019.shape[1]*(0.1)
genes_to_keep = mRNA_2019.astype(bool).sum(axis=1)[mRNA_2019.astype(bool).sum(axis=1) > threshold].index
mRNA_2019 = mRNA_2019.loc[genes_to_keep]

Let us see how genes and sample distribution change after this filtering

In [ ]:
plot_violin_metrics(mRNA_2019,1)

In [ ]:
plot_violin_metrics(mRNA_2019,0)

## Dimensionality Reduction

In [ ]:
mRNA_2019_ordered = mRNA_2019[mutated_samples.tolist() + non_mutated_samples.tolist()]
mRNA_2019_ordered_log = np.log1p(mRNA_2019_ordered)
mRNA_2019_scaled = standard(mRNA_2019_ordered_log)

In [ ]:
plot_PCA_tsne_umap(mRNA_2019_ordered,mutated_samples,non_mutated_samples,'PCA and TSNE for raw data', categories=['Mutated','Non Mutated'])

In [ ]:
plot_PCA_tsne_umap(mRNA_2019_ordered_log,mutated_samples,non_mutated_samples,'PCA and TSNE for log data', categories=['Mutated','Non Mutated'])

In [ ]:
plot_PCA_tsne_umap(mRNA_2019_scaled,mutated_samples,non_mutated_samples,'PCA and TSNE for scaled data', categories=['Mutated','Non Mutated'])

These plots give an idea of the separability of mutated and non mutated samples. In three different cases:

Raw bulk RNAseq data
Log transformed (log(1+x)) bulk RNAseq data
Standard log transformed bulk RNAseq data
We can see that data seems not very separable in both the 3 different dimensionalty reduction techniques. Despite that, the number of dimensions is very reduced and we can hope that the data would be separable in higher dimensions. This will be assessed by classifier performance.

In [ ]:
mRNA_2019.to_csv('enrichment/CCLE_2019_filtered.csv')

In [ ]:
X = mRNA_2019 
y = mut_all_2019.Variant_Type

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X.T, y,
    test_size=0.2,         
    stratify=y,            
    random_state=42        
)
X_train = X_train.T 
X_test = X_test.T

In [ ]:
samples_train_2019 = [col for col in X_train.columns if col[-1] == '9'] 
samples_train_2012 = [col for col in X_train.columns if col[-1] == 'd'] 

samples_test_2019 = [col for col in X_test.columns if col[-1] == '9'] 
samples_test_2012 = [col for col in X_test.columns if col[-1] == 'd'] 

X_train, y_train = X_train[samples_train_2019 + samples_train_2012], y_train[samples_train_2019 + samples_train_2012]
X_test, y_test = X_test[samples_test_2019 + samples_test_2012], y_test[samples_test_2019 + samples_test_2012]

In [ ]:
print("Train:",X_train.shape)
print("Test:",X_test.shape)